# Demo — Self-Attention is Permutation Invariant

Companion to [Positional Encoding Part 1](https://tanulsingh.github.io/blog/positional-encoding-absolute).

**No exercises here.** Run top-to-bottom and watch self-attention produce identical outputs (up to permutation) for two different word orderings — the problem positional encoding solves.

## Setup

In [1]:
import torch
import torch.nn.functional as F

torch.manual_seed(0)

## A minimal self-attention

No multi-head, no masks. Just Q, K, V projections and scaled dot-product attention.

In [2]:
def self_attention(x, W_q, W_k, W_v):
    """x: (seq_len, d_model) -> (seq_len, d_model)"""
    Q = x @ W_q
    K = x @ W_k
    V = x @ W_v
    d_k = Q.shape[-1]
    scores = Q @ K.T / (d_k ** 0.5)
    weights = F.softmax(scores, dim=-1)
    return weights @ V

## The experiment: 'dog bites man' vs 'man bites dog'

Three random embeddings, two orderings. Run attention on each and compare per-token outputs.

In [3]:
d_model = 8
e_dog   = torch.randn(d_model)
e_bites = torch.randn(d_model)
e_man   = torch.randn(d_model)

W_q = torch.randn(d_model, d_model)
W_k = torch.randn(d_model, d_model)
W_v = torch.randn(d_model, d_model)

x_A = torch.stack([e_dog, e_bites, e_man])   # 'dog bites man'
x_B = torch.stack([e_man, e_bites, e_dog])   # 'man bites dog'

out_A = self_attention(x_A, W_q, W_k, W_v)
out_B = self_attention(x_B, W_q, W_k, W_v)

print('out_A (dog, bites, man):')
print(out_A)
print('\nout_B (man, bites, dog):')
print(out_B)

out_A (dog, bites, man):
tensor([[-1.4719, -3.8387, -3.9765, -1.9757,  1.6286, -2.1635,  6.6189, -1.4426],
        [ 1.1710, -0.4263, -1.4974, -0.4003,  0.4218, -1.9000, -0.5898,  1.3591],
        [ 1.1554, -0.3908, -1.5021, -0.3905,  0.3919, -1.8442, -0.5646,  1.2555]])

out_B (man, bites, dog):
tensor([[ 1.1554, -0.3908, -1.5021, -0.3905,  0.3919, -1.8442, -0.5646,  1.2555],
        [ 1.1710, -0.4263, -1.4974, -0.4003,  0.4218, -1.9000, -0.5898,  1.3591],
        [-1.4719, -3.8387, -3.9765, -1.9757,  1.6286, -2.1635,  6.6189, -1.4426]])


In [4]:
# out_A index -> token in 'dog bites man':  0=dog, 1=bites, 2=man
# out_B index -> token in 'man bites dog':  0=man, 1=bites, 2=dog
# Same token should produce same output regardless of position.
assert torch.allclose(out_A[0], out_B[2], atol=1e-6), "dog's output should match across orderings"
assert torch.allclose(out_A[1], out_B[1], atol=1e-6), "bites's output should match across orderings"
assert torch.allclose(out_A[2], out_B[0], atol=1e-6), "man's output should match across orderings"
print('All asserts passed - outputs are identical up to permutation.')
print('Self-attention is permutation invariant.')

All asserts passed - outputs are identical up to permutation.
Self-attention is permutation invariant.


## Takeaway

Without positional information, self-attention is blind to order. 'dog bites man' and 'man bites dog' produce identical representations.

→ Continue to `01_sinusoidal_exercise.ipynb` to fix this.